# Python 可变对象、引用与拷贝

## Goal

通过可以逐格执行的实验理解赋值、浅拷贝、深拷贝、函数参数和可变默认参数。
每个实验都遵循“先预测，再运行，最后用 `assert` 验证”的方式。

## Setup

本教程只使用 Python 标准库，没有外部数据，也不依赖前面运行过的隐藏状态。

In [1]:
import copy


def inspect_value(name, value):
    print(
        f"{name:<10} value={value!r:<24} "
        f"type={type(value).__name__:<8} id={id(value)}"
    )

## Steps

### 1. 赋值不是复制

`alias = original` 只增加一个名字，两个名字仍指向同一个列表。

In [2]:
original = [[1, 2], [3, 4]]
alias = original

inspect_value("original", original)
inspect_value("alias", alias)
print("同一个对象:", original is alias)

original   value=[[1, 2], [3, 4]]         type=list     id=130397630627200
alias      value=[[1, 2], [3, 4]]         type=list     id=130397630627200
同一个对象: True


### 2. 对比浅拷贝和深拷贝

浅拷贝创建新的外层列表，但继续共享内层列表；深拷贝递归复制嵌套对象。

In [3]:
original = [[1, 2], [3, 4]]
shallow = original.copy()
deep = copy.deepcopy(original)

original[0].append(99)

print("original:", original)
print("shallow :", shallow)
print("deep    :", deep)
print("外层共享:", original is shallow)
print("浅拷贝内层共享:", original[0] is shallow[0])
print("深拷贝内层共享:", original[0] is deep[0])

original: [[1, 2, 99], [3, 4]]
shallow : [[1, 2, 99], [3, 4]]
deep    : [[1, 2], [3, 4]]
外层共享: False
浅拷贝内层共享: True
深拷贝内层共享: False


### 3. 函数收到的是对象引用

重新绑定局部变量不会改变调用者的名字，但修改可变对象本身会被调用者观察到。

In [4]:
def rebind(items):
    items = ["函数中的新列表"]
    return items


def mutate(items):
    items.append("函数添加")


values = ["原值"]
rebound = rebind(values)
print("重新绑定后:", values, rebound)

mutate(values)
print("原地修改后:", values)

重新绑定后: ['原值'] ['函数中的新列表']
原地修改后: ['原值', '函数添加']


### 4. 可变默认参数陷阱

默认参数在函数定义时创建一次，而不是每次调用时创建。

In [5]:
def append_bad(value, items=[]):  # noqa: B006 - 故意演示反例
    items.append(value)
    return items


def append_good(value, items=None):
    if items is None:
        items = []
    items.append(value)
    return items


print("错误:", append_bad("A"), append_bad("B"))
print("正确:", append_good("A"), append_good("B"))

错误: ['A', 'B'] ['A', 'B']
正确: ['A'] ['B']


## Checks

In [6]:
source = [[1], [2]]
shallow = source.copy()
deep = copy.deepcopy(source)

assert source is not shallow
assert source[0] is shallow[0]
assert source[0] is not deep[0]

source[0].append(3)
assert shallow == [[1, 3], [2]]
assert deep == [[1], [2]]
print("所有引用和拷贝检查通过。")

所有引用和拷贝检查通过。


## Next Steps

1. 把内层列表换成不可变元组，观察浅拷贝是否仍有风险。
2. 给列表中加入自定义类实例，再比较三种复制方式。
3. 在 VS Code 变量面板中观察外层和内层对象的 `id()`。